# MaplePy
### Madelung Part of Lattice Energy (MAPLE) for Crystal Structures

Computes per-site partial MAPLE values from crystal structures, following Hoppe's formalism.

**Key references:**
- Hoppe, R. (1966) *Angew. Chem. Int. Ed.* **5**, 95–106 — Madelung constants & partial Madelung constants
- Hoppe, R. (1970) *Angew. Chem. Int. Ed.* **9**, 25–34 — Coordination number, MAPLE, effective CN
- Hoppe, R. (1995) *Z. Naturforsch. A* **50**, 555–567 — MAPLE as solid-state diagnostic tool

**Three validation targets (all must pass before research use):**
1. **NaCl** — PMC(Na⁺) = PMC(Cl⁻) = 0.8738, MF = 1.7476  *(Hoppe 1966)*
2. **TiO₂ polymorphs** — rutile and anatase each within 50 kcal/mol of Hoppe's values, and within 15 kcal/mol of each other  *(Hoppe 1995, Table 5)*
3. **BaTiO₃ additivity** — MAPLE(BaTiO₃) ≈ MAPLE(BaO) + MAPLE(TiO₂), Δ < 1%  *(Hoppe 1995, p. 560)*

---
**Physical basis:** MAPLE is the electrostatic (Madelung) component of the lattice energy.
Unlike the global Madelung constant, *per-site partial MAPLE values* reveal which crystallographic
sites are electrostatically stable or strained — a long-range complement to BVS/GII diagnostics.

## 0. Setup

In [ ]:
# Run once to install dependencies
# !pip install pymatgen pandas numpy

In [ ]:
from pymatgen.core import __version__
print(__version__)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from pymatgen.core import Structure, Lattice
from pymatgen.analysis.ewald import EwaldSummation

# ── Physical constants ─────────────────────────────────────────────────────────
EV_TO_KCAL_MOL = 23.0609   # 1 eV (per particle) = 23.0609 kcal/mol (per mole)
KCAL_TO_KJ     = 4.184     # 1 kcal = 4.184 kJ
# MAPLE conversion factor: e²Nₐ / 4πε₀ in units of Å·kcal/mol
# Hoppe uses 331.81 (older constants); modern value is 332.06
MAPLE_FACTOR   = 332.06    # Å·kcal/mol

# ── Tolerances, each named once ────────────────────────────────────────────
#
# v1.1: these were previously written twice each, with different values.
#
# CHARGE_NEUTRAL_TOL was 0.5 where the CIF's own per-site states are accepted
# and 0.05 where the result is checked - ten times apart, in one function,
# neither with a reason. One value, used for both questions: a cell whose
# charges sum to more than this is not neutral, whoever assigned them.
# 0.05 e per cell is below the rounding of a single deposited oxidation state
# and above the floating-point noise of an occupancy-weighted sum.
CHARGE_NEUTRAL_TOL = 0.05      # e per unit cell

# OCCUPANCY_TOL was 1.05 in the CIF parser and 1.01 in the warning that
# follows it, so a site at 1.03 was silently accepted and never mentioned.
# One value: what the parser will accept is what the warning is about.
OCCUPANCY_TOL = 1.05           # site occupancy sum above which a site is
                               # reported as a refinement-quality issue


# ── When a site is not fully occupied ──────────────────────────────────────
#
# v1.3.2: the disorder test used to ask only whether a site carried more than
# one species. A single species at occupancy 0.5 is a split-site or
# vacancy-bearing refinement and is just as much a mean-field approximation,
# and 12 of 97 probed structures were labelled ordered on that basis.
#
# The number is a guard against arithmetic noise, not a scientific cut. On the
# 97-structure probe the site occupancy sums were 1.000, 0.993, 0.985, 0.950,
# 0.800, 0.731, 0.500, 0.250 and 0.167: nothing lies between 0.993 and 1.000,
# so any threshold in that gap classifies identically. Set just below 1 so
# that a refined deficiency of 0.7% is still called what it is.
FULL_OCCUPANCY_TOL = 0.995     # site occupancy sum at or above this is full

# ── When two sites are one crystallographic position ───────────────────────
#
# v1.3.2: below this separation, two sites are the same position split by a
# disorder model, and the Ewald sum over them is not an energy. Measured over
# the 1410-structure dataset, the four structures with any contact under 1 A
# are at 0.030, 0.160, 0.441 and 0.455 A, and the shortest genuine contact
# anywhere in the 97-structure probe is 1.008 A - itself a coordinate error.
# Nothing lies between 0.455 and 1.008, so any threshold in that gap
# classifies these identically; 0.5 A is well below the shortest real
# cation-anion distance in any oxide.
#
# NOTE this is a FLAG, not a filter. d_min_ang is written for every structure
# so the cut can be fitted from a full run rather than from four examples.
SPLIT_SITE_ANG = 0.5           # A, shortest contact below which two sites are
                               # taken to be one position, split
#
# v1.3.3: CONFIRMED against the whole dataset now that d_min_ang is written
# for every structure. The band from 0.5 to 1.0 A is EMPTY in both families,
# and the four structures below 0.5 A are the four this was fitted on.

# ── When a contact is too short to be a bond ───────────────────────────────
#
# v1.3.3: a second population, separate from the split sites. 26 perovskites
# have a shortest contact between 1.0 and 1.6 A, median residual 8 to 13%,
# median Madelung factor 2.5 against the family's 3.83: coordinate errors.
# Hopper's 2.5 detector catches 22 of them and structurally cannot see the
# rest, being a coefficient-of-variation test on 20-site Pnma cells. NdFeO3
# at 1.507 A was classified ADDITIVE at delta 0.312%.
#
# UNLIKE SPLIT_SITE_ANG THERE IS NO GAP HERE. It is a continuum, and this is
# a judgement rather than a fitted value. Measured, at each candidate cut:
#
#     threshold   flagged   already 2.5   newly flagged   of those, Additive
#       1.6 A        26          22             4                  1
#       1.7 A        37          27            10                  2
#       1.75 A       46          29            17                  4
#       1.8 A        63          29            34                 17
#
# 1.8 A starts taking structures whose short bond is plausible. 1.6 A is
# below any cation-anion distance these families produce and above every
# contact in the well-behaved bulk.
SHORT_CONTACT_ANG = 1.6        # A, shortest contact below which the geometry
                               # is reported as unphysical (a judgement)


# ── How many CIFs at once ──────────────────────────────────────────────────
#
# v1.3.4: every structure is independent and costs about 140 ms of CPU, and the
# 24 August perovskite run took 22 minutes. Measured on 84 real CIFs on a
# 2-core container: 10.15 s serial against 7.24 s on two workers, every value
# identical. Two cores caps that at 2x before pool startup, so on six workers
# with the startup spread over a full family, expect nearer 5x.
#
#   'auto'   as many workers as cores, less two, and never more than the
#            number of CIFs
#   1        serial, which is exactly what v1.3.3 did
#   an int   that many workers
#
# The serial and parallel paths call the SAME function, process_one_cif, so
# there is one definition of what processing a CIF means.
MAPLE_WORKERS = 'auto'


# ── The version, named once ────────────────────────────────────────────────
#
# v1.3.1: written into every results and summary CSV as a column, so a file
# on disk says which notebook made it. Three CSVs from three versions in one
# folder look identical otherwise, and the differences between them have been
# Z, the data directory and the cell setting.
MAPLEPY_VERSION = '1.3.6'


# ── Where the reference data lives ─────────────────────────────────────────
#
# v1.2: data_dir defaulted to '.', which is where the notebook is rather than
# where its data is. maple_library.json and common_oxidation_states.json sit
# in data/, so the oxidation-state priority chain silently dropped two levels
# for anyone whose working directory was the notebook's.
#
# Resolved once, here, and reported when it is used. A path given explicitly
# is checked: asking for a directory that does not exist is a mistake worth
# hearing about, not a reason to fall back.
def _default_data_dir():
    for _cand in ('data', '.'):
        if Path(_cand).is_dir() and any(Path(_cand).glob('*.json')):
            return _cand
    return '.'


DEFAULT_DATA_DIR = _default_data_dir()
print(f"  data directory: {Path(DEFAULT_DATA_DIR).resolve()}")

print(f"MaplePy {MAPLEPY_VERSION} loaded.")
print(f"  EV_TO_KCAL_MOL = {EV_TO_KCAL_MOL}")
print(f"  MAPLE_FACTOR   = {MAPLE_FACTOR} Å·kcal/mol  (Hoppe 1995 uses 331.81)")

---
## 1. Core Functions

In [ ]:
# 1. Core Functions (v3)
# ─────────────────────────────────────────────────────────────────────────────

import re as _re
import json
import math as _math
import warnings
warnings.filterwarnings('ignore', message='dict interface is deprecated',
                        category=DeprecationWarning)
warnings.filterwarnings('ignore', message='Issues encountered while parsing CIF',
                        category=UserWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning,
                        module='pymatgen')

# ── Helpers: safe extraction of element symbol and oxi_state ─────────────────
# pymatgen sites can hold either a Species object (ordered) or a Composition
# (disordered). These helpers handle both transparently.

def _site_element(site):
    """
    Return the majority element symbol for a site.
    For mixed-occupancy sites, returns the species with highest occupancy.
    """
    sp = site.species
    if len(sp) > 1:
        majority = max(sp.items(), key=lambda x: x[1])
        el = majority[0]
        return el.element.symbol if hasattr(el, 'element') else el.symbol
    if hasattr(sp, 'elements') and len(sp.elements) > 0:
        el = sp.elements[0]
        return el.symbol if hasattr(el, 'symbol') else str(el)
    return site.specie.element.symbol


def _site_oxi_state(site):
    """
    Return the oxidation state (float) for a site.
    For mixed-occupancy sites, returns the occupancy-weighted mean charge.
    """
    sp = site.species
    if len(sp) > 1:
        total_oxi = 0.0
        total_occ = 0.0
        for species, occ in sp.items():
            if hasattr(species, 'oxi_state'):
                total_oxi += species.oxi_state * occ
                total_occ += occ
        if total_occ > 0:
            return total_oxi / total_occ
        return 0.0
    if hasattr(sp, 'elements') and len(sp.elements) > 0:
        el = sp.elements[0]
        if hasattr(el, 'oxi_state'):
            return float(el.oxi_state)
    return float(site.specie.oxi_state)


def _is_anion_site(site):
    """
    Whether this site belongs to the anion sublattice.

    By oxidation state where the structure has them, and by element where it
    does not: load_structure asks about disorder before the states are
    assigned, and MaplePy's domain is oxides, so oxygen is the anion. Stated
    rather than assumed, because it is an assumption.
    """
    try:
        oxi = _site_oxi_state(site)
        if oxi:
            return oxi < 0
    except (AttributeError, TypeError, ValueError):
        pass
    return _site_element(site) == 'O'


def _disorder_detail(structure):
    """
    What kind of disorder this structure carries, and where: one computation.

    v1.3.3: `partial occupancy` was not enough. Making the flag see partial
    occupancy at all (v1.3.2) moved 14 oxygen-deficient refinements out of
    Hopper's oxygen non-stoichiometry class and into its disorder class, which
    describes them less well. Hopper has reserved a code for anion vacancy
    disorder since its first version and could not assign it without knowing
    which sublattice was short.

    MaplePy holds the occupancies, so MaplePy answers. Deriving this
    downstream from a reduced formula undercounts split sites, whose two
    half-occupied positions sum to an integer composition.

    Returns (kind, sublattice):
        kind        'ordered' | 'mixed occupancy' | 'partial occupancy'
                    | 'mixed and partial occupancy'
        sublattice  'none' | 'anion' | 'cation' | 'both'
    """
    mixed = False
    partial = set()
    for site in structure:
        if len(site.species) > 1:
            mixed = True
        if sum(site.species.values()) < FULL_OCCUPANCY_TOL:
            partial.add('anion' if _is_anion_site(site) else 'cation')

    sublattice = ('none' if not partial else
                  'both' if len(partial) > 1 else
                  next(iter(partial)))

    if mixed and partial:
        kind = 'mixed and partial occupancy'
    elif mixed:
        kind = 'mixed occupancy'
    elif partial:
        kind = 'partial occupancy'
    else:
        kind = 'ordered'
    return kind, sublattice


def _disorder_kind(structure):
    """The kind alone. Derived, so there is one computation."""
    return _disorder_detail(structure)[0]


def _is_disordered(structure):
    """True where the mean-field charge approximation is in use.

    Derived from _disorder_detail so there is one definition of disorder.
    """
    return _disorder_kind(structure) != 'ordered'


def _shortest_contact(structure):
    """
    The shortest interatomic distance in the cell, and the pair that makes it.

    Returns (d_ang, label_i, label_j). Any pair, not cation-anion: the fault
    this exists to catch is a same-element split site, which d_ref cannot see
    because d_ref only looks across the cation-anion boundary.
    """
    if len(structure) < 2:
        return float('inf'), None, None
    dm = structure.distance_matrix.copy()
    np.fill_diagonal(dm, np.inf)
    i, j = np.unravel_index(np.argmin(dm), dm.shape)
    return (float(dm[i, j]), _site_element(structure[int(i)]),
            _site_element(structure[int(j)]))


def _geometry_status(structure):
    """
    Whether the geometry can carry a Madelung energy, as a sentence.

    v1.3.2: MaplePy had no geometry test. Four structures in the 1410-entry
    dataset put two fractionally occupied sites within half an angstrom of
    each other, and their Ewald sums came out up to ten times the family
    median while being reported as ordinary results.

    v1.3.3: two thresholds, because the measured distribution has two
    populations - split sites below 0.5 A and coordinate errors between 1.0
    and 1.6 A, with nothing at all in between. See SPLIT_SITE_ANG and
    SHORT_CONTACT_ANG for both, and for which of them is a judgement.

    FLAG, KEEP, REPORT: this never drops a structure and never changes a
    number. It returns (d_min, flag, status): the flag for Hopper, which
    should not be matching on the prefix of a sentence, and the sentence for
    whoever reads the CSV.
    """
    d_min, el_i, el_j = _shortest_contact(structure)
    if d_min < SPLIT_SITE_ANG:
        return d_min, 'split_site', (
            f'split site: {el_i}-{el_j} contact {d_min:.3f} A, below '
            f'{SPLIT_SITE_ANG} A - the Ewald sum over these two sites is '
            f'not an energy')
    if d_min < SHORT_CONTACT_ANG:
        return d_min, 'short_contact', (
            f'short contact: {el_i}-{el_j} at {d_min:.3f} A, below '
            f'{SHORT_CONTACT_ANG} A - shorter than any cation-anion bond '
            f'these families produce, so the coordinates are suspect')
    return d_min, 'ok', f"ok: shortest contact {d_min:.3f} A"


# ── Oxidation state helpers (unchanged) ──────────────────────────────────────

def _load_oxi_json(path):
    """Load common_oxidation_states.json. {} if absent, raises if corrupt.

    v1.1: 'file is not there' and 'file is there and unreadable' are different
    facts and were both returning {}, which the priority chain below reads as
    'this source has nothing to say' and silently drops a level.
    """
    try:
        with open(path, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        return {}
    except (json.JSONDecodeError, OSError) as e:
        raise ValueError(f'{path} exists but could not be read: {e}') from e


def _load_maple_library(data_dir):
    # v1.3.1: no default. The directory is resolved once, in load_structure,
    # against DEFAULT_DATA_DIR; a second default here is a second answer to
    # the same question, and it was the wrong one.
    """Load maple_library.json. {} if absent, raises if corrupt. See above."""
    try:
        with open(Path(data_dir) / 'maple_library.json', 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        return {}
    except (json.JSONDecodeError, OSError) as e:
        raise ValueError(
            f'maple_library.json in {data_dir} could not be read: {e}') from e


def _extract_cif_oxidation_states(cif_path):
    """
    Parse _atom_type_oxidation_number loop from a CIF file.

    Mirrors Eir's extract_cif_oxidation_states() — the most reliable source
    for ICSD CIFs, which always embed oxidation states in this loop.

    Returns
    -------
    dict  {element: charge}  e.g. {'Na': 1, 'Cl': -1}, or {} if not found.
    """
    oxidation_states = {}
    try:
        with open(cif_path, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
    except FileNotFoundError:
        return {}
    except OSError as e:
        # v1.1: a CIF that cannot be read is not a CIF with no oxidation
        # states in it, and the caller treats the two identically.
        raise ValueError(f'could not read {cif_path}: {e}') from e

    in_loop      = False
    columns      = []
    symbol_col   = None
    oxi_col      = None

    for line in lines:
        ls = line.strip()

        if ls == 'loop_':
            in_loop    = False
            columns    = []
            symbol_col = None
            oxi_col    = None
            continue

        if ls.startswith('_atom_type_'):
            idx = len(columns)
            columns.append(ls)
            if ls == '_atom_type_symbol':
                symbol_col = idx
            elif ls == '_atom_type_oxidation_number':
                oxi_col  = idx
                in_loop  = True
            continue

        if in_loop and columns and not ls.startswith('_'):
            if not ls or ls.startswith('#') or ls.startswith('loop_'):
                in_loop = False
                continue
            parts = ls.split()
            if symbol_col is not None and symbol_col < len(parts):
                raw     = parts[symbol_col]
                element = _re.sub(r'[0-9+\-]+', '', raw)
                element = (element.capitalize() if len(element) > 1
                           else element.upper())
                if oxi_col is not None and oxi_col < len(parts):
                    ox_str = parts[oxi_col]
                    if ox_str not in ['?', '.']:
                        try:
                            oxidation_states[element] = round(float(ox_str))
                        except ValueError:
                            pass

    return oxidation_states


def _oxi_from_library(elements, reduced_formula, library):
    """
    Look up oxidation states by reduced formula in maple_library.json.
    Returns dict or None if formula not found or elements not fully covered.
    """
    overrides = library.get('oxidation_state_overrides', {})
    entry = overrides.get(reduced_formula)
    if entry and isinstance(entry, dict):
        if all(el in entry for el in elements):
            return {el: entry[el] for el in elements}
    return None


def _oxi_from_json_fallback(elements, oxi_json):
    """
    Attempt oxidation state assignment using common_oxidation_states.json.
    Returns dict or None if any element is ambiguous (multiple common states).
    """
    assignment = {}
    for el in elements:
        states = oxi_json.get(el, [])
        if len(states) == 1:
            assignment[el] = states[0]
        else:
            return None
    return assignment

# ── load_structure (v4) ──────────────────────────────────────────────────────
#
# KEY CHANGE from v3: pymatgen's native per-site oxidation state assignment
# is now the FIRST choice after user override. ICSD CIFs embed oxidation
# states in the _atom_site_type_symbol column (e.g., Mn3+, Mn4+), and
# pymatgen correctly assigns these per-site, preserving:
#   - Non-integer states (Pr3.8+ for mixed-valence)
#   - Different states on different sites (Mn3+ on site 1, Mn4+ on site 2)
#   - Charge neutrality
#
# The previous approach read element-level values from _atom_type_oxidation_number,
# which collapsed mixed-valence to a single state per element and broke charge
# balance for ~35% of structures. That is now a fallback, not the default.

def _parse_structures(parser):
    """
    Structures from a CifParser, in the PRIMITIVE cell.

    v1.3: this was `parser.get_structures()`, whose primitive=True default is
    deprecated. MaplePy has always worked in the primitive cell; nothing said
    so, and `parse_structures()` defaults the other way, so a mechanical
    rename at some future pymatgen release would have changed the cell
    without changing a line of intent.

    The choice is not load-bearing for the numbers, which is worth recording.
    Measured on the same CIFs both ways:

        pyrochlore ICSD 109080   primitive    22 sites  Z 2  MAPLE 10173.38
                                 conventional 88 sites  Z 8  MAPLE 10173.38
                                 d_ref 1.9298 and MF 8.4463 in both

        perovskite ICSD 109352   identical either way: 20 sites, Z 4,
                                 MAPLE 3914.11, d_ref 1.9855, MF 3.9007

    MAPLE per formula unit, d_ref and the Madelung factor are invariant; the
    cell and Z change together. What is NOT invariant is anything that counts
    sites - the per-site table gains 66 rows per pyrochlore in the
    conventional cell - so the cell is pinned rather than left to a default.
    """
    if hasattr(parser, 'parse_structures'):
        return parser.parse_structures(primitive=True)
    return parser.get_structures()


def _has_native_oxi_states(structure):
    """
    Check whether pymatgen assigned oxidation states when parsing the CIF.
    Returns True if all sites contain Species (not bare Element) objects.
    """
    from pymatgen.core.periodic_table import Species
    for site in structure:
        for sp in site.species:
            if not isinstance(sp, Species):
                return False
    return True


def _native_charge_balance(structure):
    """
    Compute the total charge using pymatgen's native per-site assignments.
    Handles mixed-occupancy sites correctly via occupancy weighting.
    """
    total = 0.0
    for site in structure:
        for sp, occ in site.species.items():
            if hasattr(sp, 'oxi_state'):
                total += sp.oxi_state * occ
    return total


def load_structure(cif_path, oxidation_states=None, data_dir=None):
    """
    Load a CIF and decorate with oxidation states.

    Mixed-occupancy sites are handled via mean-field charge averaging
    in the downstream compute_maple() function.

    Priority chain for oxidation state assignment:
    1. User-supplied oxidation_states dict          (highest priority)
    2. pymatgen native per-site assignment from CIF (preserves mixed valence,
       non-integer states, and per-site distinctions)
    3. _atom_type_oxidation_number loop in the CIF  (element-level fallback)
    4. maple_library.json formula override           (unambiguous binaries)
    5. common_oxidation_states.json fallback         (single-state elements)
    6. Raises ValueError with a helpful message      (never silently zeros)

    Parameters
    ----------
    cif_path        : str or Path
    oxidation_states: dict, optional  e.g. {'Na': 1, 'Cl': -1}
    data_dir        : str  path to folder containing maple_library.json
                      and common_oxidation_states.json

    Returns
    -------
    pymatgen Structure with oxidation states applied.
    """
    cif_path = str(cif_path)

    # v1.2: None means "wherever the data actually is"; an explicit path that
    # does not exist is a mistake rather than a reason to fall back silently.
    if data_dir is None:
        data_dir = DEFAULT_DATA_DIR
    elif not Path(data_dir).is_dir():
        raise ValueError(f'data_dir does not exist: {data_dir}')

    # ── Load CIF, catching unrecognised element labels ────────────────────
    try:
        from pymatgen.io.cif import CifParser
        parser = CifParser(cif_path, occupancy_tolerance=OCCUPANCY_TOL)
        structures = _parse_structures(parser)
        if not structures:
            raise ValueError("No valid structures parsed from CIF.")
        structure = structures[0]
    except Exception as e:
        if "not a valid Element" in str(e):
            raise ValueError(
                f"Unrecognised element label in CIF: {e}. "
                f"Common causes: D (deuterium) in neutron data, "
                f"or generic labels (M, X) for disordered sites."
            ) from e
        raise

    source = None
    disordered = _is_disordered(structure)

    if disordered:
        _kind, _where = _disorder_detail(structure)
        print(f"  Note: {_kind}"
              f"{'' if _where == 'none' else f' on the {_where} sublattice'}"
              f" detected. Using mean-field charge approximation.")

    # ── Check for unphysical occupancies ──────────────────────────────────
    for site in structure:
        occ_sum = sum(site.species.values())
        if occ_sum > OCCUPANCY_TOL:
            print(f"  WARNING: site occupancy sum = {occ_sum:.3f} "
                  f"(>1.0) at {site.species}. "
                  f"Refinement quality issue.")

    # ── Priority 1: user override ─────────────────────────────────────────
    if oxidation_states is not None:
        structure.add_oxidation_state_by_element(oxidation_states)
        source = 'user'

    # ── Priority 2: pymatgen native per-site assignment ───────────────────
    elif _has_native_oxi_states(structure):
        # pymatgen already assigned Species with oxi_states from the CIF.
        # Check charge balance to confirm they are sensible.
        native_q = _native_charge_balance(structure)
        if abs(native_q) < CHARGE_NEUTRAL_TOL:
            source = 'CIF (per-site, charge-neutral)'
        else:
            # Native states exist but are not charge-neutral.
            # This happens for genuinely non-stoichiometric compounds.
            # Still use them — they are the best available assignment.
            source = (f'CIF (per-site, Σq={native_q:+.2f})')
        # No action needed: structure already has the correct states.

    else:
        # ── pymatgen did not assign oxi states — fall back ────────────────
        # Extract element set
        elements = set()
        for site in structure:
            for sp in site.species:
                el = (sp.element.symbol if hasattr(sp, 'element')
                      else sp.symbol)
                elements.add(el)

        # ── Priority 3: CIF _atom_type loop (element-level) ──────────────
        cif_oxi = _extract_cif_oxidation_states(cif_path)

        if (cif_oxi
                and all(el in cif_oxi for el in elements)
                and any(v != 0 for v in cif_oxi.values())):
            structure.add_oxidation_state_by_element(cif_oxi)
            source = 'CIF (_atom_type, element-level fallback)'

        else:
            # ── Priority 4: maple_library.json ────────────────────────────
            library = _load_maple_library(data_dir)
            reduced = structure.composition.reduced_formula
            lib_oxi = _oxi_from_library(elements, reduced, library)

            if lib_oxi is not None:
                structure.add_oxidation_state_by_element(lib_oxi)
                source = f'maple_library.json ({reduced})'

            else:
                # ── Priority 5: common_oxidation_states.json ──────────────
                oxi_json = _load_oxi_json(
                    Path(data_dir) / 'common_oxidation_states.json'
                )
                json_oxi = _oxi_from_json_fallback(elements, oxi_json)

                if json_oxi is not None:
                    structure.add_oxidation_state_by_element(json_oxi)
                    source = 'common_oxidation_states.json'

                # ── Priority 6: informative failure ───────────────────────
                else:
                    missing = [el for el in elements
                               if el not in (cif_oxi or {})]
                    multi = [el for el in elements
                             if len(oxi_json.get(el, [])) > 1]
                    raise ValueError(
                        f"Could not determine oxidation states for: "
                        f"{missing + multi}.\n"
                        f"  CIF embedded states found for: "
                        f"{sorted(cif_oxi.keys()) if cif_oxi else 'none'}.\n"
                        f"  Elements with multiple common states: "
                        f"{multi}.\n"
                        f"  Reduced formula '{reduced}' not found in "
                        f"maple_library.json.\n"
                        f"  Solution: pass oxidation_states={{...}} "
                        f"explicitly, e.g.:\n"
                        f"    load_structure(path, oxidation_states="
                        f"{{'Mo': 6, 'O': -2}})"
                    )

    # ── Charge-balance check ──────────────────────────────────────────────
    total_charge = _native_charge_balance(structure)
    if abs(total_charge) > CHARGE_NEUTRAL_TOL:
        print(f"  WARNING: not charge-neutral "
              f"(Σq = {total_charge:+.3f} e).\n"
              f"  Source: {source}. Check oxidation states before "
              f"using MAPLE results.")
    else:
        # Build display of oxi states per element
        oxi_display = {}
        for s in structure:
            el = _site_element(s)
            oxi = _site_oxi_state(s)
            if el in oxi_display:
                # Multiple sites with different oxi states — show range
                if isinstance(oxi_display[el], list):
                    if oxi not in oxi_display[el]:
                        oxi_display[el].append(oxi)
                else:
                    if oxi != oxi_display[el]:
                        oxi_display[el] = [oxi_display[el], oxi]
            else:
                oxi_display[el] = oxi

        parts = []
        for el in sorted(oxi_display.keys()):
            v = oxi_display[el]
            if isinstance(v, list):
                vals = '/'.join(f"{x:+.1f}" if x != int(x)
                               else f"{int(x):+d}" for x in sorted(v))
                parts.append(f"{el}={vals}")
            else:
                if v != int(v):
                    parts.append(f"{el}={v:+.2f}")
                else:
                    parts.append(f"{el}={int(v):+d}")

        print(f"  Oxidation states ({source}): " + ", ".join(parts))

    return structure


# ── _find_d_ref ──────────────────────────────────────────────────────────────

def _find_d_ref(structure):
    """
    Shortest cation-anion distance in the structure.

    This is Hoppe's reference distance d₁ (or d_KA) — the normalisation
    distance on which the Madelung constant and MAPLE are based.
    """
    cat_idx = [i for i, s in enumerate(structure)
               if _site_oxi_state(s) > 0]
    ani_idx = [i for i, s in enumerate(structure)
               if _site_oxi_state(s) < 0]

    if not cat_idx or not ani_idx:
        # v1.1: this used to return min(distance from site 0 to anything),
        # which is not a cation-anion distance and may be anion-anion. d_ref
        # normalises the Madelung factor, so a wrong one is not a small error
        # in a diagnostic - it is a different quantity wearing the same name.
        raise ValueError(
            'cannot determine d_ref: the structure has '
            f'{len(cat_idx)} site(s) with positive charge and '
            f'{len(ani_idx)} with negative. Check the oxidation states.')

    d_min = np.inf
    for ci in cat_idx:
        for ai in ani_idx:
            d = structure.get_distance(ci, ai)
            if d < d_min:
                d_min = d
    return float(d_min)


# ── SG 62 origin-choice correction ───────────────────────────────────────

# ── The SG-62 origin correction, removed at v1.3.6 ─────────────────────────
#
# There was a function here, `_fix_sg62_origin`, which built a second
# structure with the 8d oxygen sublattice shifted by (1/2, 1/2, 0), computed
# the Ewald energy of both, and kept whichever was more negative. It is gone,
# and this is the record of why, because a deleted function leaves no trace
# to argue with.
#
# Measured across all 938 perovskites before removing it:
#
#     corrected        44    4.7% of the family, MAPLE moved by up to 8.1%
#     tested and kept 882
#     not applicable   11
#     declined          1
#     pyrochlores       0    it never fired on Fd-3m
#
# and on the eleven corrections examined in detail:
#
#   - both candidates are Pnma, with different interatomic distances and
#     energies 0.3 to 8.1% apart. They are two different structures, not two
#     descriptions of one, so this was a structural edit and not a setting fix
#   - the shortest interatomic contact is IDENTICAL in ten of the eleven, and
#     in ICSD 33482 the shift makes it WORSE, 1.658 A -> 1.462 A, and was
#     applied anyway because the energy was lower. A repair that never
#     improves the geometry is not repairing the geometry
#   - it fired on LaCrO3 220506, whose shortest contact is 1.008 A, and on
#     LuCrO3 138227, one of the RCrO3 series whose real fault is the B cation
#     deposited on 4a: adjusting broken coordinates rather than reporting them
#   - and on TbCrO3 85235, a sound structure agreeing with an independent 2023
#     refinement and additive at 0.056%, whose MAPLE it moved by 2.8%
#
# The selection rule was the Ewald energy, which is the quantity this study
# measures, applied to structures this study is judging.
#
# NOTHING GENERAL REPLACES IT, because there is nothing mis-expanded to
# repair: the deposited structure is the deposited structure. What does the
# work instead is v1.3.3's geometry reporting - d_min_ang, geometry_flag,
# SPLIT_SITE_ANG and SHORT_CONTACT_ANG - and Hopper's codes 2.5 and 2.7. A
# structure that cannot carry a Madelung energy is now reported as such
# rather than quietly improved.
#
# If a repair is ever wanted again, the rule to meet is the one this failed:
# apply it only where the deposited structure FAILS the geometry test and the
# candidate PASSES it. On this dataset that fires on none of the 44.


def _derive_Z(structure):
    """
    Formula units per cell, from the site counts rather than the composition.

    v1.1. `composition.get_reduced_composition_and_factor()` cannot factor a
    NON-STOICHIOMETRIC composition, so it returns Z = 1 and every per-formula
    -unit quantity is inflated by the true Z. Over the 1410-structure
    perovskite and pyrochlore dataset:

        Z = 1 entries                                   172  (12.2%)
        of those, with a fractional subscript           165  (96%)
        Z > 1 entries with a fractional subscript         0  (0%)

    Gd4Cr3.984O12 is not a metadata error; it is a refinement with a partly
    occupied Cr site, and it has four formula units in its cell like every
    other GdFeO3-type perovskite.

    Z is crystallographic, so take it from the crystallography. The number of
    SITES containing each element is an integer whatever the occupancies are,
    and Z divides all of them, so Z is their greatest common divisor. No
    composition arithmetic, no rounding of occupancies, no structure-type
    knowledge.

    Verified on 103 ICSD CIFs: identical to composition reduction on all 95
    where reduction works, and 4 rather than 1 on all 8 where it does not -
    every one a 20-site perovskite. No disagreement in the other direction.

    Falls back to the composition factor only when the structure has no sites
    to count, which cannot happen for a parsed CIF but is not worth an
    exception.
    """
    counts = {}
    for site in structure:
        for sp in site.species:
            el = sp.element.symbol if hasattr(sp, 'element') else sp.symbol
            counts[el] = counts.get(el, 0) + 1
    if not counts:
        _, z_float = structure.composition.get_reduced_composition_and_factor()
        return max(1, int(round(z_float)))
    z = 0
    for n in counts.values():
        z = _math.gcd(z, int(n))
    return max(1, z)


# ── compute_maple ────────────────────────────────────────────────────────────

def compute_maple(structure, verbose=True):
    """
    Compute partial MAPLE values for all sites in the structure.

    Uses pymatgen's EwaldSummation for the electrostatic energy.
    Per-site contributions are the row sums of the energy matrix,
    which sum to the total Ewald energy without double-counting.

    For mixed-occupancy sites, the occupancy-weighted mean charge is used.

    Sign convention
    ---------------
    pymatgen energies are negative (bound state).
    MAPLE is reported positive, following Hoppe's convention.

    Parameters
    ----------
    structure : pymatgen Structure with oxidation states
    verbose   : bool

    Returns
    -------
    dict with keys:
        total_maple_kcal, per_site, Z, d_ref, madelung_factor,
        disordered, structure
    """
    disorder_kind, partial_sublattice = _disorder_detail(structure)
    disordered   = disorder_kind != 'ordered'


    # ── SG 62 origin-choice correction ────────────────────────────────────
    # ── Ewald summation ───────────────────────────────────────────────────
    # v1.3.6: on the structure as deposited. The origin correction that used
    # to run first, and hand its summation down, is gone; see the record
    # above.
    ewald = EwaldSummation(structure)
    energy_matrix = ewald.total_energy_matrix
    per_site_eV   = energy_matrix.sum(axis=1)

    # ── Formula units per cell (Z) ────────────────────────────────────────
    Z = _derive_Z(structure)

    # ── Conversion: eV/site → kcal/mol/site ──────────────────────────────
    per_site_maple = np.abs(per_site_eV) * EV_TO_KCAL_MOL

    # ── Total MAPLE per formula unit ──────────────────────────────────────
    total_maple_kcal = np.abs(ewald.total_energy) * EV_TO_KCAL_MOL / Z

    # ── Reference distance ────────────────────────────────────────────────
    d_ref = _find_d_ref(structure)

    # ── Geometry ──────────────────────────────────────────────────────────
    # v1.3.2: measured, reported, and never used to drop anything.
    d_min, geometry_flag, geometry_status = _geometry_status(structure)

    # ── Per-site data ─────────────────────────────────────────────────────
    per_site_data = []
    for i, site in enumerate(structure):
        oxi     = _site_oxi_state(site)
        maple_i = float(per_site_maple[i])

        pmc = (maple_i * d_ref / (MAPLE_FACTOR * abs(oxi))
               ) if oxi != 0 else None

        reduced = (maple_i / (oxi ** 2) / d_ref
                   ) if oxi != 0 else None

        per_site_data.append({
            "site_index"      : i,
            "element"         : _site_element(site),
            "oxidation_state" : round(oxi, 4),
            "mixed_site"      : len(site.species) > 1,
            "frac_x"          : round(float(site.frac_coords[0]), 6),
            "frac_y"          : round(float(site.frac_coords[1]), 6),
            "frac_z"          : round(float(site.frac_coords[2]), 6),
            "maple_kcal_mol"  : round(maple_i, 3),
            "pmc"             : round(pmc, 5)     if pmc is not None else None,
            "reduced_maple"   : round(reduced, 4) if reduced is not None else None,
        })

    # ── Madelung factor ───────────────────────────────────────────────────
    #
    # v1.1: THE CONVENTION, WRITTEN DOWN. The mean charges below are means
    # over SITES, not over stoichiometry and not weighted by multiplicity.
    # For ABO3 with two trivalent cations the two agree; for a 2+/4+
    # perovskite the site mean is 3, and for the pyrochlore 16d/16c/48f/8b
    # counts it is a different average again. Nothing here is changed,
    # because the classification and every published figure rest on this
    # definition - but it is a choice, it was undocumented, and it scales the
    # axis on which the Z = 1 population was identified. It should be checked
    # against Hoppe's own normalisation before any future version moves it.
    try:
        oxi_states = [_site_oxi_state(s) for s in structure]
        z_pos = [z for z in oxi_states if z > 0]
        z_neg = [z for z in oxi_states if z < 0]
        if not z_pos or not z_neg:
            madelung_factor = None
        else:
            mean_zp = np.mean(z_pos)
            mean_zn = abs(np.mean(z_neg))
            madelung_factor = (total_maple_kcal * d_ref
                               / (MAPLE_FACTOR * mean_zp * mean_zn))
    except (TypeError, ValueError, ZeroDivisionError, AttributeError):
        madelung_factor = None

    result = {
        "total_maple_kcal" : round(total_maple_kcal, 2),
        "per_site"         : per_site_data,
        "Z"                : Z,
        "d_ref"            : round(d_ref, 4),
        "madelung_factor"  : (round(madelung_factor, 4)
                              if madelung_factor is not None else None),
        "disordered"       : disordered,
        "disorder_status"  : disorder_kind,
        "partial_sublattice": partial_sublattice,
        "d_min_ang"        : round(d_min, 4),
        "geometry_flag"    : geometry_flag,
        "geometry_status"  : geometry_status,
        "structure"        : structure,
    }

    if verbose:
        comp = structure.composition.reduced_formula
        sg   = structure.get_space_group_info()[0]
        _where = ('' if partial_sublattice in ('none', None)
                  else f' ({partial_sublattice})')
        tag  = (f"  [MEAN-FIELD: {disorder_kind}{_where}]"
                if disordered else "")
        print(f"\n{'─'*60}")
        print(f"  Compound  : {comp}  ({sg}){tag}")
        print(f"  Sites     : {len(structure)}  |  Z = {Z}")
        print(f"  d_ref     : {d_ref:.4f} Å")
        print(f"  Geometry  : {geometry_status}")
        mf_str = f"{madelung_factor:.4f}" if madelung_factor is not None else "n/a"
        print(f"  MF        : {mf_str}")
        print(f"  MAPLE     : {total_maple_kcal:.1f} kcal/mol  "
              f"({total_maple_kcal * KCAL_TO_KJ:.1f} kJ/mol)")
        print(f"{'─'*60}")

    return result


# ── results_to_dataframe ─────────────────────────────────────────────────────

def results_to_dataframe(maple_result):
    """Convert compute_maple() output to a tidy pandas DataFrame."""
    df = pd.DataFrame(maple_result["per_site"])

    comp = maple_result["structure"].composition.reduced_formula
    sg   = maple_result["structure"].get_space_group_info()[0]
    df["compound"]        = comp
    df["space_group"]     = sg
    df["Z"]               = maple_result["Z"]
    df["d_ref_ang"]       = maple_result["d_ref"]
    df["madelung_factor"] = maple_result["madelung_factor"]
    df["total_maple"]     = maple_result["total_maple_kcal"]
    df["disordered"]      = maple_result["disordered"]
    df["disorder_status"] = maple_result.get("disorder_status")
    df["partial_sublattice"] = maple_result.get("partial_sublattice")
    df["d_min_ang"]       = maple_result.get("d_min_ang")
    df["geometry_flag"]   = maple_result.get("geometry_flag")
    df["geometry_status"] = maple_result.get("geometry_status")

    cols = ["compound", "space_group", "Z", "d_ref_ang", "madelung_factor",
            "total_maple", "disordered", "disorder_status",
            "partial_sublattice", "d_min_ang", "geometry_flag",
            "geometry_status", "site_index", "element",
            "oxidation_state", "mixed_site",
            "frac_x", "frac_y", "frac_z",
            "maple_kcal_mol", "pmc", "reduced_maple"]
    return df[cols]

---
## 2. Validation

**All three checks must pass before using MaplePy for research.**
These are the exact values Hoppe reports; any significant deviation indicates
an implementation error.

### 2.1  NaCl — Partial Madelung constants

**Expected (Hoppe 1966):**
- PMC(Na⁺) = PMC(Cl⁻) = **0.8738** (rock salt has commutative sublattices → equal PMC)
- Total MF = **1.7476**
- The symmetry equality PMC(Na) = PMC(Cl) is an exact result, not a tolerance question.

In [ ]:
# NaCl: Fm-3m, a = 5.6402 Å (experimental ambient, ICSD)
nacl = Structure.from_spacegroup(
    "Fm-3m",
    Lattice.cubic(5.6402),
    ["Na", "Cl"],
    [[0.0, 0.0, 0.0], [0.5, 0.5, 0.5]]
)
nacl.add_oxidation_state_by_element({"Na": 1, "Cl": -1})

result_nacl = compute_maple(nacl)
df_nacl = results_to_dataframe(result_nacl)
print(df_nacl[["element","oxidation_state","maple_kcal_mol","pmc","reduced_maple"]].to_string(index=False))

In [ ]:
# ── Validation checks ──────────────────────────────────────────────────────────
pmc_na = df_nacl[df_nacl["element"] == "Na"]["pmc"].mean()
pmc_cl = df_nacl[df_nacl["element"] == "Cl"]["pmc"].mean()
mf     = result_nacl["madelung_factor"]

TARGET_PMC = 0.8738
TARGET_MF  = 1.7476
TOL        = 0.005  # 0.5%

print("── NaCl Validation ───────────────────────────────────────────────────────")
print(f"  PMC(Na⁺)   = {pmc_na:.4f}  target {TARGET_PMC}  Δ = {abs(pmc_na-TARGET_PMC):.4f}")
print(f"  PMC(Cl⁻)   = {pmc_cl:.4f}  target {TARGET_PMC}  Δ = {abs(pmc_cl-TARGET_PMC):.4f}")
print(f"  MF (total) = {mf:.4f}  target {TARGET_MF}  Δ = {abs(mf-TARGET_MF):.4f}")
print(f"  Symmetry   |PMC_Na - PMC_Cl| = {abs(pmc_na-pmc_cl):.2e}  (must be ~0)")

pass_na  = abs(pmc_na - TARGET_PMC) < TOL
pass_cl  = abs(pmc_cl - TARGET_PMC) < TOL
pass_mf  = abs(mf     - TARGET_MF)  < TOL
pass_sym = abs(pmc_na - pmc_cl)     < 1e-4

overall = all([pass_na, pass_cl, pass_mf, pass_sym])
print(f"\n  PMC(Na) : {'✓ PASS' if pass_na  else '✗ FAIL'}")
print(f"  PMC(Cl) : {'✓ PASS' if pass_cl  else '✗ FAIL'}")
print(f"  MF      : {'✓ PASS' if pass_mf  else '✗ FAIL'}")
print(f"  Symmetry: {'✓ PASS' if pass_sym else '✗ FAIL'}")
print(f"\n  {'✅  NaCl VALIDATION PASSED' if overall else '❌  NaCl VALIDATION FAILED — check implementation'}")

### 2.2  TiO₂ polymorphs — Near-degenerate modifications

**Expected (Hoppe 1995, Table 5):**
| Polymorph | MAPLE (kcal/mol) |
|-----------|:---------------:|
| Rutile    | 3254.6          |
| Anatase   | 3260.7          |
| Brookite  | 3253.7          |

Brookite is given for reference only; the checks below compute rutile and anatase.

The key result: despite radically different geometries, MAPLE values differ by only
~7 kcal/mol. This is a non-trivial consequence of the additivity theorem and is what
MAPLE is designed to reveal.

In [ ]:
# Rutile TiO₂: P4₂/mnm (No. 136), a = 4.5933 Å, c = 2.9592 Å
# Ti: 2a (0,0,0); O: 4f (u,u,0) with u = 0.3048
rutile = Structure.from_spacegroup(
    136,
    Lattice.tetragonal(4.5933, 2.9592),
    ["Ti", "O"],
    [[0.0, 0.0, 0.0], [0.3048, 0.3048, 0.0]]
)
rutile.add_oxidation_state_by_element({"Ti": 4, "O": -2})

# Anatase TiO₂: I4₁/amd (No. 141), a = 3.7845 Å, c = 9.5143 Å
# Ti: 4a; O: 8e (0,0,z) with z = 0.2082
anatase = Structure.from_spacegroup(
    141,
    Lattice.tetragonal(3.7845, 9.5143),
    ["Ti", "O"],
    [[0.0, 0.0, 0.0], [0.0, 0.0, 0.2082]]
)
anatase.add_oxidation_state_by_element({"Ti": 4, "O": -2})

print("Rutile  :", rutile.composition,  " | sites:", len(rutile))
print("Anatase :", anatase.composition, " | sites:", len(anatase))

In [ ]:
result_rutile  = compute_maple(rutile,  verbose=True)
result_anatase = compute_maple(anatase, verbose=True)

In [ ]:
maple_r = result_rutile["total_maple_kcal"]
maple_a = result_anatase["total_maple_kcal"]
delta   = abs(maple_r - maple_a)

print("── TiO₂ Validation ───────────────────────────────────────────────────────")
print(f"  MAPLE(rutile)  = {maple_r:.1f}  kcal/mol  (Hoppe: 3254.6)")
print(f"  MAPLE(anatase) = {maple_a:.1f}  kcal/mol  (Hoppe: 3260.7)")
print(f"  |Δ| rutile–anatase = {delta:.1f} kcal/mol")

# Tolerances: 50 kcal/mol absolute (allows for slightly different CIF sources),
# and < 15 kcal/mol inter-polymorph (the near-degeneracy is the key result)
pass_r    = abs(maple_r - 3254.6) < 50
pass_a    = abs(maple_a - 3260.7) < 50
pass_diff = delta < 15

overall = all([pass_r, pass_a, pass_diff])
print(f"\n  Rutile  vs Hoppe 3254.6  : {'✓ PASS' if pass_r    else '✗ FAIL'}  (Δ={abs(maple_r-3254.6):.1f})")
print(f"  Anatase vs Hoppe 3260.7  : {'✓ PASS' if pass_a    else '✗ FAIL'}  (Δ={abs(maple_a-3260.7):.1f})")
print(f"  Polymorphs within 15 kcal: {'✓ PASS' if pass_diff else '✗ FAIL'}  (Δ={delta:.1f})")
print(f"\n  {'✅  TiO₂ VALIDATION PASSED' if overall else '❌  TiO₂ VALIDATION FAILED'}")

### 2.3  BaTiO₃ — Additivity theorem

**Hoppe's additivity theorem:** MAPLE(polynary) = Σ MAPLE(binary constituents)

**Expected (Hoppe 1995, p. 560):**
- MAPLE(BaO) + MAPLE(rutile-TiO₂) = 4092 kcal/mol
- MAPLE(BaTiO₃, cubic) = 4095 kcal/mol
- Δ = 3 kcal/mol = **0.07%**

This near-exact agreement across the binary → ternary transition is the fundamental
validation of MAPLE as a comparative structural tool.

In [ ]:
# BaO: Fm-3m (rock salt), a = 5.5391 Å
bao = Structure.from_spacegroup(
    "Fm-3m",
    Lattice.cubic(5.5391),
    ["Ba", "O"],
    [[0.0, 0.0, 0.0], [0.5, 0.5, 0.5]]
)
bao.add_oxidation_state_by_element({"Ba": 2, "O": -2})

# Cubic BaTiO₃: Pm-3m (No. 221), a = 4.0019 Å
# Ba: 1a (0,0,0)  Ti: 1b (½,½,½)  O: 3c (½,½,0)
batibo3 = Structure.from_spacegroup(
    "Pm-3m",
    Lattice.cubic(4.0019),
    ["Ba", "Ti", "O"],
    [[0.0, 0.0, 0.0], [0.5, 0.5, 0.5], [0.5, 0.5, 0.0]]
)
batibo3.add_oxidation_state_by_element({"Ba": 2, "Ti": 4, "O": -2})

print("BaO:    ", bao.composition,    "| sites:", len(bao))
print("BaTiO₃: ", batibo3.composition, "| sites:", len(batibo3))

In [ ]:
result_bao    = compute_maple(bao,    verbose=True)
result_batio3 = compute_maple(batibo3, verbose=True)

In [ ]:
maple_bao    = result_bao["total_maple_kcal"]
maple_tio2   = result_rutile["total_maple_kcal"]   # from validation 2.2
maple_batio3 = result_batio3["total_maple_kcal"]

maple_sum  = maple_bao + maple_tio2
delta_abs  = abs(maple_batio3 - maple_sum)
delta_pct  = 100 * delta_abs / maple_batio3

print("── BaTiO₃ Additivity Validation ──────────────────────────────────────────")
print(f"  MAPLE(BaO)             = {maple_bao:.1f}  kcal/mol")
print(f"  MAPLE(TiO₂, rutile)    = {maple_tio2:.1f}  kcal/mol")
print(f"  Binary sum             = {maple_sum:.1f}  kcal/mol  (Hoppe: 4092)")
print(f"  MAPLE(BaTiO₃, cubic)   = {maple_batio3:.1f}  kcal/mol  (Hoppe: 4095)")
print(f"  Residual |Δ|           = {delta_abs:.1f} kcal/mol = {delta_pct:.3f}%")

pass_add = delta_pct < 1.0
print(f"\n  Additivity < 1%  : {'✓ PASS' if pass_add else '✗ FAIL'}  ({delta_pct:.3f}%)")
print(f"  {'✅  ADDITIVITY VALIDATION PASSED' if pass_add else '❌  ADDITIVITY VALIDATION FAILED'}")

---
## 3. Single CIF Usage

In [ ]:
# Replace with your CIF path and oxidation states if not embedded in the CIF.
#
# structure = load_structure(
#     "path/to/structure.cif",
#     oxidation_states={"Ca": 2, "O": -2}   # omit if CIF has them
# )
# result = compute_maple(structure)
# df     = results_to_dataframe(result)
# display(df)
# df.to_csv("maple_output.csv", index=False)
#
# ── Demo on already-computed NaCl result ───────────────────────────────────────
df_demo = results_to_dataframe(result_nacl)
display(df_demo)

---
## 4. Additivity Checker

General-purpose function for any polynary compound.

In [ ]:
def check_additivity(polynary_result, binary_results_and_coeffs, verbose=True):
    """
    Check Hoppe's additivity theorem: MAPLE(polynary) ≈ Σ coeff_i × MAPLE(binary_i)

    Parameters
    ----------
    polynary_result : dict from compute_maple()
    binary_results_and_coeffs : list of (result_dict, float) tuples
        e.g. [(result_bao, 1), (result_tio2, 1)] for BaTiO₃ = BaO + TiO₂

    Returns
    -------
    dict  with keys: maple_polynary, maple_binary_sum, delta_abs, delta_pct
    """
    maple_poly = polynary_result["total_maple_kcal"]
    maple_sum  = sum(c * r["total_maple_kcal"] for r, c in binary_results_and_coeffs)

    delta_abs = abs(maple_poly - maple_sum)
    delta_pct = 100 * delta_abs / maple_poly if maple_poly else 0

    if verbose:
        comp = polynary_result["structure"].composition.reduced_formula
        print(f"Additivity: {comp}")
        for r, c in binary_results_and_coeffs:
            bc = r["structure"].composition.reduced_formula
            print(f"  {c:+.0f} × MAPLE({bc}) = {c * r['total_maple_kcal']:.1f} kcal/mol")
        print(f"  Binary sum   = {maple_sum:.1f} kcal/mol")
        print(f"  Polynary     = {maple_poly:.1f} kcal/mol")
        print(f"  |Δ|          = {delta_abs:.1f} kcal/mol = {delta_pct:.3f}%")
        status = ("✓ GOOD (<1%)" if delta_pct < 1.0 else
                  "⚠ MARGINAL (1–2%)" if delta_pct < 2.0 else
                  "✗ POOR (>2%)")
        print(f"  Additivity   : {status}")

    return {"maple_polynary": maple_poly, "maple_binary_sum": maple_sum,
            "delta_abs": delta_abs, "delta_pct": delta_pct}

In [ ]:
# Example: BaTiO₃ using results already computed in validation
check_additivity(result_batio3, [(result_bao, 1), (result_rutile, 1)])

---
## 5. Batch Processing

In [ ]:
# 5. Batch Processing (v3)
# ─────────────────────────────────────────────────────────────────────────────

# ── Where the last run went ────────────────────────────────────────────────
#
# v1.3.1: Section 8 used to carry a hard-coded path to somebody's Downloads
# folder, and a run and the plots of that run were connected only by the
# reader retyping it. Both entry points record where they wrote.
LAST_OUTPUT_DIR = None      # Path, set by batch_maple() and launch_maple_gui()

# The per-structure summary, one definition. Section 8 and Hopper both open
# maple_summary.csv and read these names.
SUMMARY_COLS = ["compound", "space_group", "Z", "d_ref_ang",
                "madelung_factor", "total_maple", "disordered",
                "disorder_status", "partial_sublattice", "d_min_ang",
                "geometry_flag", "geometry_status",
                "source_cif", "maplepy_version"]


def _write_outputs(dfs, failed, output_dir):
    """
    Write the three CSVs a MAPLE run produces, and record where they went.

    v1.3.1: batch_maple() wrote maple_results.csv and maple_failed.csv but not
    maple_summary.csv, while the GUI wrote all three. Section 8 and Hopper
    both look for the summary first, so a batch run could not be plotted or
    decomposed and nothing said why. Written in one place now, so the two
    entry points cannot drift again.

    Returns (master, summary); both empty frames if nothing was processed.
    """
    global LAST_OUTPUT_DIR

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if dfs:
        master = pd.concat(dfs, ignore_index=True)
        master["maplepy_version"] = MAPLEPY_VERSION
        results_csv = output_dir / 'maple_results.csv'
        master.to_csv(results_csv, index=False)
        print(f"\nSaved {len(master)} rows → {results_csv}")

        summary = (master[SUMMARY_COLS]
                   .drop_duplicates(subset="source_cif")
                   .reset_index(drop=True))
        summary_csv = output_dir / 'maple_summary.csv'
        summary.to_csv(summary_csv, index=False)
        print(f"Saved {len(summary)} structures → {summary_csv}")
    else:
        master = pd.DataFrame()
        summary = pd.DataFrame()
        print("\nNo structures processed successfully.")

    if failed:
        failed_df = pd.DataFrame(failed, columns=['cif_file', 'error'])
        failed_csv = output_dir / 'maple_failed.csv'
        failed_df.to_csv(failed_csv, index=False)
        print(f"Saved {len(failed)} failures → {failed_csv}")

    LAST_OUTPUT_DIR = output_dir
    return master, summary


def process_one_cif(cif_path, oxidation_states=None, data_dir=None,
                    verbose=False):
    """
    Load, compute and tabulate ONE CIF. The one definition.

    v1.3.4: batch_maple and the GUI each had their own copy of this sequence,
    which is two things to keep in step and two things to put in a worker.
    Returns only picklable data, so it can run in another process:

        (name, df or None, error or None, tag, console output)

    `tag` is the mean-field marker for the progress line, so the caller can
    print without reaching back into the result.
    """
    import contextlib as _ctx
    import io as _io
    from pathlib import Path as _P

    name = _P(cif_path).name
    # v1.3.4: the console output is CAPTURED and handed back, not left to
    # wherever this happens to run. load_structure prints which source the
    # oxidation states came from, unconditionally, and that line is the only
    # record of the provenance anywhere - it is in no CSV. A worker process
    # writes it to a pipe nobody reads, so a parallel run would have quietly
    # stopped telling you. Captured here, printed by the caller, identical
    # either way.
    _said = _io.StringIO()
    try:
        with _ctx.redirect_stdout(_said):
            s = load_structure(cif_path, oxidation_states=oxidation_states,
                               data_dir=data_dir)
            result = compute_maple(s, verbose=verbose)
            df = results_to_dataframe(result)
        df["source_cif"] = name
        return (name, df, None, (" [MF]" if result["disordered"] else ""),
                _said.getvalue())
    except Exception as e:
        return name, None, str(e), "", _said.getvalue()


def _resolve_workers(workers, n_items):
    """How many processes to actually start. Never more than there is work."""
    if workers in (None, 1, 0):
        return 1
    if workers == 'auto':
        try:
            import os as _os
            workers = max(1, (_os.cpu_count() or 2) - 2)
        except Exception:
            return 1
    try:
        workers = int(workers)
    except (TypeError, ValueError):
        print(f"  MAPLE_WORKERS = {workers!r} is not a number or 'auto'; "
              f"running serially.")
        return 1
    return max(1, min(workers, n_items))


def _map_cifs(paths, oxidation_states_map, data_dir, verbose, workers):
    """
    process_one_cif over every path, in order, serially or in a pool.

    v1.3.4: the only place that decides between the two, and both branches
    call the same function. A pool that will not start is reported in full
    and the run continues serially - an optimisation with an exact serial
    equivalent should never be the thing that stops the science.
    """
    from pathlib import Path as _P
    n = _resolve_workers(workers, len(paths))
    jobs = [(p, (oxidation_states_map or {}).get(_P(p).stem)) for p in paths]

    if n > 1:
        try:
            from joblib import Parallel, delayed
        except ImportError as e:
            print(f"  joblib is not installed ({e}), so MAPLE_WORKERS "
                  f"cannot be honoured. Running serially. `pip install "
                  f"joblib` for the parallel path.")
            n = 1
    if n > 1:
        print(f"  processing on {n} workers")
        try:
            try:
                # joblib >= 1.3 streams results in order, so progress is
                # visible during the run rather than only at the end
                out = Parallel(n_jobs=n, backend='loky',
                               return_as='generator')(
                    delayed(process_one_cif)(p, oxi, data_dir, False)
                    for p, oxi in jobs)
            except TypeError:
                out = Parallel(n_jobs=n, backend='loky')(
                    delayed(process_one_cif)(p, oxi, data_dir, False)
                    for p, oxi in jobs)
            for item in out:
                yield item
            return
        except Exception as e:
            print(f"  the worker pool failed to run ({type(e).__name__}: "
                  f"{e}). Falling back to serial processing; the results are "
                  f"the same either way, only slower.")

    for p, oxi in jobs:
        yield process_one_cif(p, oxi, data_dir, verbose)


def batch_maple(cif_dir, output_dir=None, oxidation_states_map=None,
                data_dir=None, verbose=False, workers=None):
    """
    Process all CIFs in a directory.

    Parameters
    ----------
    cif_dir : str or Path
        Directory containing CIF files.
    output_dir : str or Path, optional
        Directory for output CSVs. If None, creates a timestamped
        outputs_YYYYMMDD_HHMMSS folder inside cif_dir.
    oxidation_states_map : dict  {filename_stem: {element: oxi_state}}
        For CIFs without embedded oxidation states.
    data_dir : str or None
        Folder holding maple_library.json and common_oxidation_states.json.
        None (v1.3.1) means DEFAULT_DATA_DIR, resolved at load. It used to be
        '.', which was passed on explicitly and so overrode that resolution:
        every batch run silently lost two levels of the oxidation-state
        priority chain.
    verbose : bool
        If True, print full MAPLE output per structure.

    Returns
    -------
    pd.DataFrame, one row per site per structure
    """
    cif_dir   = Path(cif_dir)
    cif_files = sorted(cif_dir.glob("*.cif"))
    if not cif_files:
        print(f"No CIF files found in {cif_dir}")
        return pd.DataFrame()

    # ── Create output directory ───────────────────────────────────────────
    if output_dir is None:
        from datetime import datetime
        timestamp  = datetime.now().strftime('%Y%m%d_%H%M%S')
        output_dir = cif_dir / f'outputs_{timestamp}'
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"Processing {len(cif_files)} CIFs in {cif_dir}")
    print(f"Output directory: {output_dir}")
    dfs, failed = [], []

    # v1.3.4: one worker function, run serially or on a pool by _map_cifs.
    for name, df_i, err, tag, said in _map_cifs(
            [str(p) for p in cif_files], oxidation_states_map, data_dir,
            verbose, MAPLE_WORKERS if workers is None else workers):
        if said:
            print(said, end='')
        if err is not None:
            failed.append((name, err))
            print(f"  ✗ {name:45s}  {err}")
            continue
        dfs.append(df_i)
        print(f"  ✓ {name:45s}  "
              f"MAPLE = {df_i['total_maple'].iloc[0]:.1f} kcal/mol{tag}")

    # ── Save results ──────────────────────────────────────────────────────
    master, _summary = _write_outputs(dfs, failed, output_dir)

    # ── Summary ───────────────────────────────────────────────────────────
    n_total = len(cif_files)
    n_ok    = len(dfs)
    n_fail  = len(failed)
    n_ordered    = sum(1 for d in dfs
                       if not d["disordered"].iloc[0]) if dfs else 0
    n_disordered = n_ok - n_ordered

    print(f"\n{'='*60}")
    print(f"  Total CIFs        : {n_total}")
    print(f"  Processed (OK)    : {n_ok}  "
          f"({n_ordered} ordered, {n_disordered} mean-field)")
    print(f"  Failed            : {n_fail}")
    print(f"{'='*60}")

    return master


# Usage:
# df = batch_maple(Path("/path/to/cifs"))
# df = batch_maple(Path("/path/to/cifs"), output_dir=Path("/custom/output"))

---
## 6. Application: MAPLE vs BVS for Alkaline Earth Oxides

The primary research motivation: test whether MAPLE site potentials correlate with
the systematic BVS deviations in the MgO → CaO → SrO → BaO series (Figure 1,
Reeves-McLaren 2026, *Dalton Trans.*).

**Hypothesis:** If MAPLE(M²⁺) scales linearly with cation radius while BVS deviation
also scales linearly (R² = 0.996 established), the BVS functional form is missing
long-range electrostatic contributions rather than being fundamentally wrong in its
short-range parameterisation. This would sharpen the "parameter revision required"
conclusion and point towards a specific correction strategy.

In [ ]:
# Rock-salt MO structures from experimental ambient ICSD parameters
ae_params = {
    "MgO": {"a": 4.2112, "cation": "Mg", "z": 2, "r_shannon": 0.720},
    "CaO": {"a": 4.8105, "cation": "Ca", "z": 2, "r_shannon": 1.000},
    "SrO": {"a": 5.1408, "cation": "Sr", "z": 2, "r_shannon": 1.180},
    "BaO": {"a": 5.5391, "cation": "Ba", "z": 2, "r_shannon": 1.350},
}

ae_results = {}
for name, p in ae_params.items():
    s = Structure.from_spacegroup(
        "Fm-3m", Lattice.cubic(p["a"]),
        [p["cation"], "O"], [[0,0,0],[0.5,0.5,0.5]]
    )
    s.add_oxidation_state_by_element({p["cation"]: p["z"], "O": -2})
    ae_results[name] = compute_maple(s, verbose=False)

    r = p["r_shannon"]
    m = ae_results[name]["total_maple_kcal"]
    d = ae_results[name]["d_ref"]
    print(f"  {name:4s}  r(M²⁺)={r:.3f}Å  d_ref={d:.4f}Å  MAPLE={m:.1f} kcal/mol")

In [ ]:
# Per-site MAPLE for the cation series
rows = []
for name, result in ae_results.items():
    p   = ae_params[name]
    df_i = pd.DataFrame(result["per_site"])
    maple_cat = df_i[df_i["element"] == p["cation"]]["maple_kcal_mol"].mean()
    maple_ani = df_i[df_i["element"] == "O"]["maple_kcal_mol"].mean()
    pmc_cat   = df_i[df_i["element"] == p["cation"]]["pmc"].mean()
    rows.append({
        "oxide"          : name,
        "cation"         : p["cation"],
        "shannon_r_ang"  : p["r_shannon"],
        "d_ref_ang"      : result["d_ref"],
        "madelung_factor": result["madelung_factor"],
        "maple_total"    : result["total_maple_kcal"],
        "maple_cation"   : round(maple_cat, 2),
        "maple_anion"    : round(maple_ani, 2),
        "pmc_cation"     : round(pmc_cat, 4),
    })

df_ae = pd.DataFrame(rows)
print(df_ae.to_string(index=False))

# Save for merge with BVS deviation data from Eir output
# df_ae.to_csv("ae_oxide_maple.csv", index=False)

In [ ]:
# Quick correlation check: does MAPLE(M²⁺) correlate with Shannon radius?
from numpy.polynomial import polynomial as P

r_vals    = df_ae["shannon_r_ang"].values
maple_cat = df_ae["maple_cation"].values
maple_tot = df_ae["maple_total"].values

# Linear fit
coeffs_cat = np.polyfit(r_vals, maple_cat, 1)
coeffs_tot = np.polyfit(r_vals, maple_tot, 1)

# R²
def r_squared(y, y_fit):
    ss_res = np.sum((y - y_fit)**2)
    ss_tot = np.sum((y - y.mean())**2)
    return 1 - ss_res/ss_tot

r2_cat = r_squared(maple_cat, np.polyval(coeffs_cat, r_vals))
r2_tot = r_squared(maple_tot, np.polyval(coeffs_tot, r_vals))

print("── MAPLE vs Shannon radius (alkaline earth oxides) ───────────────────────")
print(f"  MAPLE(cation) ~ radius:  slope={coeffs_cat[0]:.1f} kcal/mol/Å  R²={r2_cat:.4f}")
print(f"  MAPLE(total)  ~ radius:  slope={coeffs_tot[0]:.1f} kcal/mol/Å  R²={r2_tot:.4f}")
print()
print("Compare with BVS result from Reeves-McLaren (2026):")
print("  BVS deviation ~ radius:  slope ≈ -30 %/Å  R² = 0.996")
print()
print("If MAPLE(cation) also scales linearly (R² >> 0), this supports the")
print("hypothesis that BVS is missing long-range electrostatic contributions.")

---
## 7. GUI: Load and Process Real CIFs

Tkinter folder picker → `batch_maple()` on every CIF in the folder → three CSVs in a timestamped `outputs_` folder beside them.

**Oxidation states:** set `OXIDATION_STATES` at the top of the cell below before launching.
ICSD CIFs for simple binary oxides and halides usually embed oxidation states and
the dict can be left empty (`{}`). For CIFs without embedded states, or where
pymatgen's auto-assignment is unreliable (transition metals, mixed valence),
provide explicit values keyed by element symbol.

The output CSV uses the same column schema as the programmatic results above,
so it can be merged directly with Eir output for BVS/MAPLE comparison.

In [ ]:
# 7. GUI: Load and Process Real CIFs (v3)
# ─────────────────────────────────────────────────────────────────────────────
#
# Tkinter FOLDER picker → batch_maple on every CIF in it → three CSVs in a
# timestamped outputs_ folder beside them.
#
# v1.3.5: this used to be a FILE picker, so every CIF had to be selected by
# hand, and the cell then carried its own copy of the output directory, the
# progress lines, the summary table and the totals - all of which batch_maple
# already did. The GUI is now a folder picker in front of batch_maple, so
# there is one implementation of the batch and the button just points at it.
#
# Three output files are created in an 'outputs' folder alongside the CIFs:
#   maple_results.csv  — full site-level data (every site, every structure)
#   maple_summary.csv  — one row per structure (compound, SG, MAPLE, etc.)
#   maple_failed.csv   — every failed CIF with its error message


# ── User-editable oxidation state overrides ───────────────────────────────────
# Leave empty {} to use CIF-embedded states or pymatgen auto-assignment.
# Add entries only for elements where you want to override, e.g.:
#   OXIDATION_STATES = {"Ba": 2, "O": -2}

OXIDATION_STATES = {}

# ── Data directory ────────────────────────────────────────────────────────────
# Folder containing common_oxidation_states.json (and shannon_radii.json).
#
# v1.3.1: None means DEFAULT_DATA_DIR, resolved once at load (Section 0). This
# was "." and was passed on explicitly, which overrode that resolution, so the
# GUI never read data/ however the rest of the notebook was configured.
DATA_DIR = None

print("Oxidation state overrides:",
      OXIDATION_STATES if OXIDATION_STATES else
      "(none — will use CIF or auto-assign)")


# ── GUI function ──────────────────────────────────────────────────────────────

import tkinter as tk
from tkinter import filedialog
import traceback

# ── Where the CIFs are ────────────────────────────────────────────────────────
# Set this to a folder and no dialog opens. Useful for a second family in the
# same session, and immune to the hidden-dialog problem below.
MAPLE_CIF_DIR = None       # e.g. r"path/to/cifs"


def launch_maple_gui(cif_dir=None):
    """
    Pick a folder of CIFs and run batch_maple on it.

    v1.3.5: a FOLDER, not a hand-picked list of files, and the work itself is
    batch_maple rather than a second copy of it. Everything the batch does -
    the worker pool, the three CSVs, LAST_OUTPUT_DIR, the summary - comes with
    it, because it is the same function.

    Pass cif_dir, or set MAPLE_CIF_DIR above, to skip the dialog.
    """
    cif_dir = cif_dir or MAPLE_CIF_DIR

    if not cif_dir:
        # A second Tk dialog in one kernel can open BEHIND the browser, and
        # the cell then waits on a window nobody can see. The notice goes out
        # before the call that blocks, so a hang is diagnosable rather than
        # mysterious; parent, update() and lift() make it far less likely.
        print("Opening a folder picker. If nothing appears, look BEHIND this "
              "window. Set MAPLE_CIF_DIR above to skip the dialog entirely.")
        root = tk.Tk()
        root.withdraw()
        root.update()
        root.attributes('-topmost', True)
        root.lift()
        root.focus_force()
        cif_dir = filedialog.askdirectory(
            title="Select the folder containing your CIF files",
            parent=root
        )
        root.update()
        root.destroy()

    if not cif_dir:
        print("No folder selected.")
        return None

    cif_dir = Path(cif_dir)
    n_cifs = len(sorted(cif_dir.glob("*.cif")))
    print(f"Folder: {cif_dir}")
    print(f"  {n_cifs} CIF file(s)")
    if not n_cifs:
        print("  Nothing to do: no *.cif in that folder. Note that this does "
              "not look in subfolders.")
        return None

    oxi = ({p.stem: OXIDATION_STATES for p in cif_dir.glob("*.cif")}
           if OXIDATION_STATES else None)

    master = batch_maple(cif_dir, oxidation_states_map=oxi,
                         data_dir=DATA_DIR)

    if master is not None and len(master):
        summary = (master[SUMMARY_COLS]
                   .drop_duplicates(subset="source_cif")
                   .reset_index(drop=True))
        print("\nSummary (per structure):")
        display(summary)
        return master
    return None


# ── Launch ────────────────────────────────────────────────────────────────────
df_loaded = launch_maple_gui()


### Interpreting the GUI output

**Output files** (saved to a timestamped `outputs_YYYYMMDD_HHMMSS/` folder alongside the CIF files):

| File | Contents |
|------|----------|
| `maple_results.csv` | Full site-level data: one row per crystallographic site per structure. Use this for per-site analysis, BVS/MAPLE cross-comparison, and detailed diagnostics. |
| `maple_summary.csv` | One row per structure: compound, space group, Z, d_ref, MF, total MAPLE, disorder flag. Use this for database-scale statistics and the additivity analysis. |
| `maple_failed.csv` | One row per failed CIF: filename and error message. Use this for failure mode classification. |

**Column definitions (site-level, `maple_results.csv`):**

| Column | Meaning |
|--------|---------|
| `total_maple` | MAPLE per formula unit [kcal/mol]. For isostructural compounds, lower = longer bonds = larger cation. |
| `madelung_factor` | Dimensionless MF. Constant within a structure type (1.7476 for all rock salts). Deviations signal a topology change. |
| `d_ref_ang` | Shortest cation–anion distance [Å]. The reference distance on which PMC is normalised. |
| `pmc` | Partial Madelung constant. Geometrically invariant within a structure type; site-specific in lower-symmetry structures. |
| `reduced_maple` | MAPLE / z² / d_ref. Strips charge and distance dependence; enables cross-structure, cross-valence comparison. |
| `disordered` | `True` if any site is shared by more than one species or partly occupied (`disorder_status` says which). MAPLE calculated using mean-field charge approximation. |
| `mixed_site` | `True` for individual sites with mixed occupancy. The `element` column reports the majority species; `oxidation_state` reports the occupancy-weighted mean charge. |

**Diagnostic plots** (saved to the same folder, as PNG):

| Plot | What it shows | What to look for |
|------|---------------|------------------|
| `plot_1_processing_overview.png` | Bar chart of ordered successes, mean-field successes, and failures by type. | Overall processing yield; ratio of ordered to disordered structures in the ICSD subset. |
| `plot_2_maple_vs_complexity.png` | Total MAPLE *vs.* atoms per formula unit, coloured by ordered/mean-field. | Strong linear correlation expected for well-behaved ionic compounds. Outliers flag unusual electrostatics or incorrect oxidation states. |
| `plot_3_madelung_factor_distribution.png` | Histogram of Madelung factors with NaCl reference line (1.7476). | Clustering by structure type; spread within a cluster indicates structural variation or errors. |

In [ ]:
# 8. Diagnostic Plots
# ─────────────────────────────────────────────────────────────────────────────
#
# Run this cell AFTER a run: either the GUI (Section 7) or batch_maple()
# (Section 5). It reads maple_summary.csv, maple_results.csv and
# maple_failed.csv from that run and writes three PNGs beside them.
#
# Nothing to set. Both entry points record where they wrote in
# LAST_OUTPUT_DIR, and this cell reads it. To plot an OLDER run instead, put
# its CIF folder in CIF_DIR below.

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from pathlib import Path

# ── Which run to plot ─────────────────────────────────────────────────────────
#
# v1.3.1: this line used to be a literal path to another machine's Downloads
# folder, and the error it raised named the wrong cause - it reported that the
# folder held no outputs_* when the folder itself was not there.
#
# Leave CIF_DIR as None to plot the run made in this kernel. Set it to the
# folder holding your CIFs (the one containing outputs_* subdirectories), or
# to an outputs_* folder itself, to plot an earlier run.
CIF_DIR = None
# ──────────────────────────────────────────────────────────────────────────────

if CIF_DIR:
    cif_dir = Path(CIF_DIR)
    if not cif_dir.is_dir():
        raise FileNotFoundError(
            f"CIF_DIR is not a folder on this machine: {cif_dir}\n"
            f"Set CIF_DIR to a folder that exists here, or set it to None to "
            f"plot the run made in this kernel.")
    runs = sorted(cif_dir.glob('outputs_*'), reverse=True)
    if runs:
        output_dir = runs[0]
    elif (cif_dir / 'outputs').is_dir():
        output_dir = cif_dir / 'outputs'        # a pre-timestamp run
    elif (cif_dir / 'maple_summary.csv').exists():
        output_dir = cif_dir                    # CIF_DIR IS the outputs folder
    else:
        raise FileNotFoundError(
            f"{cif_dir} exists but holds no run: no outputs_* folder, no "
            f"outputs/ folder and no maple_summary.csv.\n"
            f"Run Section 7 or batch_maple() first.")
elif globals().get('LAST_OUTPUT_DIR') is not None:
    output_dir = Path(LAST_OUTPUT_DIR)
else:
    raise RuntimeError(
        "No run to plot. Either run the GUI (Section 7) or batch_maple() "
        "(Section 5) in this kernel, or set CIF_DIR above to the folder "
        "holding an earlier run's CIFs.")

print(f"Plotting: {output_dir}")

# ── Sheffield brand colours ───────────────────────────────────────────────────
SHEFFIELD = {
    'brand_violet' : '#440099',
    'dark_violet'  : '#24125E',
    'electric'     : '#7000FF',
    'powder_violet': '#A78CFA',
    'pale_violet'  : '#C8BBFF',
    'teal'         : '#005A8F',
    'aqua'         : '#00BBCC',
    'coral'        : '#E7004C',
    'flamingo'     : '#FF6371',
    'peach'        : '#FF9664',
    'peak_green'   : '#005750',
    'spearmint'    : '#3BD4AE',
    'midnight'     : '#131E29',
}

# ── Global plot styling ───────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family'      : 'sans-serif',
    'font.size'        : 11,
    'axes.labelsize'   : 13,
    'axes.titlesize'   : 14,
    'figure.dpi'       : 150,
    'savefig.dpi'      : 300,
    'savefig.bbox'     : 'tight',
    'axes.linewidth'   : 1.0,
    'xtick.direction'  : 'in',
    'ytick.direction'  : 'in',
    'xtick.major.width': 1.0,
    'ytick.major.width': 1.0,
})


# ── Load data ─────────────────────────────────────────────────────────────────
summary_csv = output_dir / 'maple_summary.csv'
failed_csv  = output_dir / 'maple_failed.csv'
results_csv = output_dir / 'maple_results.csv'

if not summary_csv.exists():
    raise FileNotFoundError(
        f"No maple_summary.csv found in {output_dir}.\n"
        f"That folder is a run directory but the summary is missing: the run "
        f"processed no structures, or it was made by MaplePy before v1.3.1, "
        f"whose batch_maple() did not write one."
    )

df_summary = pd.read_csv(summary_csv)
df_results = pd.read_csv(results_csv) if results_csv.exists() else None

if failed_csv.exists():
    df_failed = pd.read_csv(failed_csv)
else:
    df_failed = pd.DataFrame(columns=['cif_file', 'error'])

n_ordered    = (~df_summary['disordered']).sum()
n_disordered = df_summary['disordered'].sum()
n_failed     = len(df_failed)
n_total      = n_ordered + n_disordered + n_failed

print(f"Loaded: {len(df_summary)} successful structures, "
      f"{n_failed} failures, {n_total} total")


# ═════════════════════════════════════════════════════════════════════════════
# PLOT 1: Success / Failure Overview
# ═════════════════════════════════════════════════════════════════════════════

fig1, ax1 = plt.subplots(figsize=(8, 4))

# Classify failures
if len(df_failed) > 0:
    conditions = [
        df_failed['error'].str.contains('Mixed-occupancy', na=False),
        df_failed['error'].str.contains('not a valid Element', na=False),
        df_failed['error'].str.contains('Unrecognised element', na=False),
        df_failed['error'].str.contains('Invalid cif', na=False),
        df_failed['error'].str.contains('charge-neutral', na=False)
            | df_failed['error'].str.contains('NoneType', na=False),
    ]
    labels_fail = ['Mixed occupancy', 'Invalid element (D, M, ...)',
                   'Invalid CIF', 'Charge imbalance']

    # Count each failure type
    fail_counts = {}
    accounted = pd.Series(False, index=df_failed.index)
    for cond, label in zip(conditions, labels_fail):
        mask = cond & ~accounted
        fail_counts[label] = mask.sum()
        accounted = accounted | cond
    fail_counts['Other'] = (~accounted).sum()
    # Remove zero-count categories
    fail_counts = {k: v for k, v in fail_counts.items() if v > 0}
else:
    fail_counts = {}

# Build bar data
categories = []
counts     = []
colours    = []

categories.append('Ordered')
counts.append(int(n_ordered))
colours.append(SHEFFIELD['brand_violet'])

categories.append('Mean-field')
counts.append(int(n_disordered))
colours.append(SHEFFIELD['powder_violet'])

fail_colours = [SHEFFIELD['coral'], SHEFFIELD['peach'],
                SHEFFIELD['flamingo'], SHEFFIELD['aqua'],
                SHEFFIELD['spearmint']]
for i, (label, count) in enumerate(fail_counts.items()):
    categories.append(label)
    counts.append(count)
    colours.append(fail_colours[i % len(fail_colours)])

bars = ax1.barh(categories, counts, color=colours, edgecolor='white',
                linewidth=0.5)

# Add count labels on bars
for bar, count in zip(bars, counts):
    if count > 0:
        ax1.text(bar.get_width() + max(sum(counts)*0.01, 0.5),
                 bar.get_y() + bar.get_height()/2,
                 str(count), va='center', ha='left', fontsize=10,
                 color=SHEFFIELD['midnight'])

ax1.set_xlabel('Number of CIF files')
ax1.set_title('MaplePy Processing Results')
ax1.invert_yaxis()
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Add percentage annotation
pct_ok = 100 * (n_ordered + n_disordered) / n_total if n_total > 0 else 0
ax1.text(0.98, 0.02, f'{pct_ok:.1f}% processed successfully',
         transform=ax1.transAxes, ha='right', va='bottom',
         fontsize=10, color=SHEFFIELD['dark_violet'], style='italic')

fig1.tight_layout()
fig1.savefig(output_dir / 'plot_1_processing_overview.png')
print(f"Saved: {output_dir / 'plot_1_processing_overview.png'}")
plt.show()


# ═════════════════════════════════════════════════════════════════════════════
# PLOT 2: Total MAPLE vs formula complexity
# ═════════════════════════════════════════════════════════════════════════════

if df_results is not None and len(df_summary) > 1:

    # Count atoms per formula unit from site-level data
    atoms_per_fu = (df_results.groupby('source_cif')
                    .apply(lambda g: len(g) / g['Z'].iloc[0],
                           include_groups=False)
                    .reset_index(name='atoms_per_fu'))

    df_plot = df_summary.merge(atoms_per_fu, on='source_cif', how='left')

    fig2, ax2 = plt.subplots(figsize=(7, 6))

    # Ordered vs mean-field
    mask_ord = ~df_plot['disordered']
    mask_dis = df_plot['disordered']

    ax2.scatter(df_plot.loc[mask_ord, 'atoms_per_fu'],
                df_plot.loc[mask_ord, 'total_maple'],
                c=SHEFFIELD['brand_violet'], s=25, alpha=0.7,
                edgecolors='white', linewidth=0.3,
                label=f'Ordered ({mask_ord.sum()})', zorder=3)

    if mask_dis.any():
        ax2.scatter(df_plot.loc[mask_dis, 'atoms_per_fu'],
                    df_plot.loc[mask_dis, 'total_maple'],
                    c=SHEFFIELD['powder_violet'], s=25, alpha=0.7,
                    edgecolors='white', linewidth=0.3, marker='D',
                    label=f'Mean-field ({mask_dis.sum()})', zorder=3)

    ax2.set_xlabel('Atoms per formula unit')
    ax2.set_ylabel('Total MAPLE / kcal mol$^{-1}$')
    ax2.set_title('MAPLE vs formula complexity')
    ax2.legend(frameon=False, fontsize=10)
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)

    fig2.tight_layout()
    fig2.savefig(output_dir / 'plot_2_maple_vs_complexity.png')
    print(f"Saved: {output_dir / 'plot_2_maple_vs_complexity.png'}")
    plt.show()

else:
    print("Plot 2 skipped: need >1 structure in maple_summary.csv")


# ═════════════════════════════════════════════════════════════════════════════
# PLOT 3: Madelung factor distribution
# ═════════════════════════════════════════════════════════════════════════════

if len(df_summary) > 1:

    mf_vals = df_summary['madelung_factor'].dropna()

    fig3, ax3 = plt.subplots(figsize=(7, 5))

    ax3.hist(mf_vals, bins=min(50, max(10, len(mf_vals)//5)),
             color=SHEFFIELD['teal'], edgecolor='white', linewidth=0.5,
             alpha=0.85)

    ax3.set_xlabel('Madelung factor')
    ax3.set_ylabel('Count')
    ax3.set_title('Distribution of Madelung factors')
    ax3.spines['top'].set_visible(False)
    ax3.spines['right'].set_visible(False)

    # Annotate common structure types if recognisable peaks appear
    ax3.axvline(x=1.7476, color=SHEFFIELD['coral'], linestyle='--',
                linewidth=1, alpha=0.7)
    ax3.text(1.7476, ax3.get_ylim()[1]*0.95, ' NaCl\n (1.748)',
             fontsize=8, color=SHEFFIELD['coral'], va='top')

    fig3.tight_layout()
    fig3.savefig(output_dir / 'plot_3_madelung_factor_distribution.png')
    print(f"Saved: {output_dir / 'plot_3_madelung_factor_distribution.png'}")
    plt.show()

else:
    print("Plot 3 skipped: need >1 structure in maple_summary.csv")


print(f"\nAll plots saved to: {output_dir}")

---
## 9. Notes, Limitations & Future Work

### Limitations

**Disorder:** a site shared by more than one species, or partly occupied, carries the
occupancy-weighted mean charge of its species (a mean-field approximation; Section 1),
and the structure is flagged `disordered`, with the kind of disorder and the sublattice
it is on. The approximation smooths the electrostatic landscape of the ordered
configurations it stands for.

**Molecular anions:** The additivity theorem breaks down for compounds containing
molecular-like anions (SO₄²⁻, PO₄³⁻, CO₃²⁻ etc.) where covalent bonding within
the anion is significant. Hoppe (1995) treats these using MAPLE "increments" derived
from well-characterised reference structures (e.g. MAPLE("SO₃") from sulphates).

**Reference distance:** `d_ref` is computed as the shortest cation–anion distance.
For direct quantitative comparison with Hoppe's tabulated values, the same CIF source
should be used. Differences in CIF geometry (different measurement conditions, year)
explain most discrepancies between calculated and Hoppe's MAPLE values.

### Integration with Eir
- Eir reads CIFs with gemmi; MaplePy reads them with pymatgen.
- Integration path: add a `--maple` flag to Eir CLI that calls `compute_maple()`
  and appends MAPLE columns to the existing site-level CSV output.
- Column names follow Eir conventions: `lowercase_snake_case`, `_kcal_mol` suffix.

### Future extensions
- MAPLE-WAGII: occupancy-weighted partial MAPLE for disordered structures
- MEFIR (Mean Fictive Ionic Radii, Hoppe & Meyer 1976): back-calculated from PMC values
- Database-scale batch processing with ICSD export
- BVS vs MAPLE correlation analysis for systematic parameter inadequacy diagnosis

### Citation
When publishing results calculated with MaplePy, cite the release you used (its DOI
is in `CITATION.cff`) and the paper describing the method (Reeves-McLaren and Gross,
*J. Appl. Cryst.*, under review), and name the version (`MAPLEPY_VERSION`). The
formalism is that of Hoppe (1966, *Angew. Chem. Int. Ed.* **5**, 95) and Hoppe
(1995, *Z. Naturforsch. A* **50**, 555).